# LG-FedAvg Server

> The core abstraction for `LG-FedAvg` server: [https://arxiv.org/pdf/2001.01523](https://arxiv.org/pdf/2001.01523)

In [ ]:
#| default_exp servers.lgfedavg

In [ ]:
#| hide
from nbdev.showdoc import *
from fastcore.test import *

In [ ]:
#| export
from fastcore.utils import *
from fastcore.all import *
import os

import copy
from tqdm import tqdm
from loguru import logger

import numpy as np
import pandas as pd

import torch
import torch.nn.functional as F
from torch.utils.data import DataLoader

from fedai.client_selector import BaseClientSelector
from fedai.data import prepare_dl
from fedai.models import create_model
from fedai.optimizers import get_optimizer

from fedai.servers.base_server import BaseServer
from fedai.utils.registery import AlgorithmRegistry
from fedai.core import get_clean_kwargs
from fedai.optimizers.custom_optimizers import PerturbedGradientDescent

<torch._C.Generator>

In [ ]:
#| export
@AlgorithmRegistry.register_server("lgfedavg")
class ServerLGFedAvg(BaseServer):
    def __init__(self,
                 cfg,
                 client_selector,
                 client_cls,
                 criterion,
                 fds,
                 writer= None,
                 device= None,
                 **kwargs
                 ):  
                 
        super().__init__(cfg, client_selector, client_cls, criterion, fds, writer, device, **kwargs) 
        self.global_model = copy.deepcopy(self.model.head)
                 

In [ ]:
#| export
@patch
def client_fn(self: ServerLGFedAvg, id, comm_round, client_state):

    if (comm_round == 1 and client_state == {}) or client_state == {}:
        client_state['local_model'] = copy.deepcopy(self.model.backbone.state_dict())
        client_state['global_head'] = copy.deepcopy(self.global_model.state_dict())

    local_model = create_model(self.cfg).backbone
    local_model.load_state_dict(client_state['local_model'])
    client_state['local_model'] = local_model

    global_head = create_model(self.cfg).head
    global_head.load_state_dict(client_state['global_head'])
    client_state['global_head'] = global_head

    
    kwargs = get_clean_kwargs(self.cfg.optimizer)
    kwargs.pop("cls", None)
    kwargs.pop("lr", None)
    g1 = {"params": client_state['local_model'].parameters(), "lr": self.cfg.algorithm.local_lr, **kwargs}
    g2 = {"params": client_state['global_head'].parameters(), "lr": self.cfg.algorithm.global_lr, **kwargs}
    optimizer = get_optimizer(self.cfg)([ g1, g2 ])
    optimizer.load_state_dict(client_state['optimizer']) if 'optimizer' in client_state else None
    for state in optimizer.state.values():
        for k, v in state.items():
            if isinstance(v, torch.Tensor):
                state[k] = v.to(self.device)

    client_state['optimizer'] = optimizer

    train_loader = prepare_dl(self.cfg, id, self.fds, train=True, distributed=False)
    test_loader = prepare_dl(self.cfg, id, self.fds, train=False, distributed=False)
    client = self.client_cls(id= id, 
                             cfg= self.cfg,
                             train_loader= train_loader,
                             test_loader= test_loader,
                             state= client_state,
                             criterion= self.criterion,
                             device= self.device,
                             t= comm_round)
    client.logger = self.logger
    return client


### Aggregation


In [ ]:
#| export
@patch
def aggregate(self: ServerLGFedAvg, lst_active_ids, comm_round, len_clients_ds):
    m_t = sum(len_clients_ds.values())
    
    with torch.no_grad():
        global_model = None
        
        for id in lst_active_ids:
            client_state_dict = self.state_mgr.get_state(id).get('global_head', None)

            if global_model is None:
                global_model = {k: torch.zeros_like(v) for k, v in client_state_dict.items()}

            n_k = len_clients_ds[id]
            weight = n_k / m_t
            for key in global_model.keys():
                global_model[key].add_(client_state_dict[key], alpha=weight)

        self.model.head.load_state_dict(global_model)
        self.global_model.load_state_dict(global_model)
        
        for id in lst_active_ids:
            self.state_mgr.set_state(id, {
                    'local_model': self.state_mgr.get_state(id).get('local_model', None),
                    'global_head': self.global_model.state_dict(),
                    'optimizer': self.state_mgr.get_state(id).get('optimizer', None),
            })
        

In [ ]:
#| hide
import nbdev; nbdev.nbdev_export()